# Lab 1: The building blocks

Plan for the three labs that build the assignment:

| Lab | Layer | What you build |
|---|---|---|
| 1 (this one) | Building blocks: `langchain-openai` + Pydantic | LLM calls, messages, structured output, **retry and fallback** |
| 2 | Orchestration: `langgraph` | State, reducers, nodes, edges and the **loop**, with no LLM at all |
| 3 | The project | Planner, executor loop, Word builder, FastAPI |

The final project lives in `doc_agent.py`, `doc_tools.py` and `api.py`, just as the Sidekick lives in `sidekick.py`, `sidekick_tools.py` and `app.py`.

**Why LangGraph and not `create_agent` or a Deep Agent?** The assignment grades *your* planning design. `create_agent` hands you the tool loop and a Deep Agent hands you a whole harness. Our task has a fixed shape (plan, then loop over the plan, then build) so we draw that graph ourselves. It stays small, predictable and easy to explain.

**Before you run this**

You need an `OPENAI_API_KEY` in a `.env` file next to this notebook:

```
OPENAI_API_KEY=sk-...
```
The model name lives in one constant near the top so you can change it in one place.

In [1]:
import os
import time
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

load_dotenv(override=True)

MODEL1 = "gpt-6-luna"   # change this in one place if you use another model
MODEL2 = "gpt-5.4-nano"  # change this in one place if you use another model

## A first model call

`ChatOpenAI` is the abstraction around the OpenAI API. `invoke` is the key method: you give it a prompt and get a message back.

In [4]:
llm = ChatOpenAI(model=MODEL1, api_key=os.getenv("OPENAI_API_KEY"))
reply = llm.invoke("In one sentence, what does it mean for an AI agent to plan its own work?")
print(reply.content)

It means the AI agent decides what steps to take, in what order, and how to adjust them to achieve its goal.


## Messages

A system message sets the role and rules. A human message carries the request. You can pass the message classes or plain dictionaries, the result is the same. Our planner and executor will both use a system message to give the model its job.

In [5]:
messages = [
    SystemMessage("You are a project manager. Answer in exactly one sentence."),
    HumanMessage("What is a good first step when a client request is vague?"),
]
print(llm.invoke(messages).content)

Ask the client a few focused questions to clarify the desired outcome, priorities, and constraints before estimating or starting the work.


## Structured output: let Pydantic describe the answer

Our program cannot do much with a paragraph of prose. It needs fields: a title, a list of assumptions, a list of tasks. So instead of asking for JSON and parsing it by hand, we describe the answer with Pydantic classes and call `with_structured_output`. The model is forced into that shape, and what comes back is a **validated Pydantic object**, not a dictionary you have to check.

These are the three schemas the whole project uses.

In [6]:
class Task(BaseModel):
    """One item in the agent's TODO list."""
    id: int
    section: str = Field(description="Section name, for example 'Risks'")
    goal: str = Field(description="What this section must achieve")


class Plan(BaseModel):
    """What the planner returns."""
    doc_type: str = Field(description="proposal, project plan, SOP, business report, etc.")
    title: str
    assumptions: list[str] = Field(description="Assumptions for missing information and resolved conflicts")
    tasks: list[Task] = Field(description="4 to 7 tasks, one per document section")


class Section(BaseModel):
    """One written section of the document."""
    heading: str
    paragraphs: list[str]
    bullets: list[str]

In [8]:
planner = llm.with_structured_output(Plan)

user_query = """Prepare a proposal for a client. I don't know their industry or budget. 
           "They want it ready fast but also very detailed. Include risks and a timeline."""

system_message = """You plan business documents. State an assumption for every gap and every conflict.
               "Create 4 to 7 tasks, one per section."""

plan = planner.invoke([
    ("system", system_message),
    ("user", user_query),
])
print(plan.model_dump_json(indent=2))


{
  "doc_type": "Client proposal",
  "title": "Industry-Neutral Client Proposal",
  "assumptions": [
    "The client's industry, organization, needs, and desired outcomes were not provided; the proposal will use adaptable, industry-neutral language and mark client-specific details for confirmation.",
    "No budget or pricing information was provided; investment will be presented as a placeholder or subject to discovery, with no figures invented.",
    "The request to be ready fast while also being very detailed is treated as a need for a promptly usable proposal with substantive detail; use clear sections and identify items that require follow-up rather than delaying for unavailable information.",
    "No project start date, duration, or availability information was provided; the timeline will use relative phases and depend on timely client input and approval."
  ],
  "tasks": [
    {
      "id": 1,
      "section": "Executive Summary and Objectives",
      "goal": "Frame the opportun

In [9]:
print(type(plan))
print("Title:", plan.title, "|", plan.doc_type)
for a in plan.assumptions:
    print("  assumption:", a)
for t in plan.tasks:
    print(f"  {t.id}. {t.section}: {t.goal}")

<class '__main__.Plan'>
Title: Industry-Neutral Client Proposal | Client proposal
  assumption: The client's industry, organization, needs, and desired outcomes were not provided; the proposal will use adaptable, industry-neutral language and mark client-specific details for confirmation.
  assumption: No budget or pricing information was provided; investment will be presented as a placeholder or subject to discovery, with no figures invented.
  assumption: The request to be ready fast while also being very detailed is treated as a need for a promptly usable proposal with substantive detail; use clear sections and identify items that require follow-up rather than delaying for unavailable information.
  assumption: No project start date, duration, or availability information was provided; the timeline will use relative phases and depend on timely client input and approval.
  1. Executive Summary and Objectives: Frame the opportunity, proposed value, and intended outcomes in adaptable la

Look at what you got: `plan.title` and `plan.tasks[0].section` are real attributes. That is the agent's TODO list, produced by the model.

## Things go wrong, so we add retry and fallback (our mandatory improvement)

Any LLM call can fail: a rate limit, a timeout, a reply that breaks the schema. If one failure crashes the whole agent, nothing works end to end. The fix has two parts:

1. **Retry** the same model a couple of times, since most failures are momentary.
2. **Fall back** to a second model if the first keeps failing.

LangChain has a one-line way to do this, shown first. Every runnable has `.with_retry()` and `.with_fallbacks()`.

In [10]:
primary = llm.with_structured_output(Plan).with_retry(stop_after_attempt=2)
backup = ChatOpenAI(model=MODEL2).with_structured_output(Plan)   # use a different model here if you have one
resilient_planner = primary.with_fallbacks([backup])

print(resilient_planner.invoke("Create a plan for a short meeting-minutes document about a sprint review.").title)

Sprint Review — Meeting Minutes


That is compact, but it fails silently, and for the video you want to *show* each failed attempt. So in the project we write the same idea by hand in a small function called `ask`. It tries the primary model twice, then the fallback model once, prints every failure, and raises one clear `LLMError` if everything fails.

In [11]:
PRIMARY_MODEL = MODEL1
FALLBACK_MODEL = MODEL2   # put a different or cheaper model here if you have one
RETRY_DELAY = 1            # seconds between failed attempts


class LLMError(Exception):
    """Raised when every attempt, including the fallback model, has failed."""


def ask(schema, system: str, user: str, retries: int = 3):
    """Call the LLM and return a validated `schema` object. Retry, then fall back."""
    models = [PRIMARY_MODEL] * (retries - 1) + [FALLBACK_MODEL]
    last_error = "unknown"
    for attempt, model_name in enumerate(models, start=1):
        try:
            structured = ChatOpenAI(model=model_name).with_structured_output(schema)
            result = structured.invoke([("system", system), ("user", user)])
            if result is None:
                raise ValueError("model returned no structured output")
            if not isinstance(result, schema):
                result = schema.model_validate(result)
            return result
        except Exception as error:
            last_error = f"{type(error).__name__}: {error}"
            print(f"[ask] attempt {attempt}/{len(models)} on '{model_name}' failed -> {last_error[:200]}")
            if attempt < len(models):
                time.sleep(RETRY_DELAY)
    raise LLMError(f"All {len(models)} attempts failed. Last error: {last_error}")

### Test 1: the normal path

In [12]:
plan = ask(Plan, "You plan business documents. Create 4 to 7 tasks.", "Create a project plan for launching a mobile app in 3 months with a 5-person team.")
print(plan.title, "|", len(plan.tasks), "tasks")

Three-Month Mobile App Launch Plan | 5 tasks


### Test 2: break it on purpose

First a wrong primary model name: you should see two failed attempts and then success on the fallback. Then both wrong: you should get an `LLMError`. This is also a good bug story for the debugging part of your video.

In [13]:
good_primary, good_fallback = PRIMARY_MODEL, FALLBACK_MODEL

PRIMARY_MODEL = "this-model-does-not-exist"
try:
    print(ask(Plan, "You plan business documents.", "Plan a short SOP for onboarding a new employee.").title)
finally:
    PRIMARY_MODEL = good_primary

PRIMARY_MODEL, FALLBACK_MODEL = "bad-1", "bad-2"
try:
    ask(Plan, "You plan business documents.", "Plan a short SOP for onboarding a new employee.")
except LLMError as error:
    print("Got LLMError as expected:", str(error)[:120])
finally:
    PRIMARY_MODEL, FALLBACK_MODEL = good_primary, good_fallback

[ask] attempt 1/3 on 'this-model-does-not-exist' failed -> OpenAIModelNotFoundError: Error code: 404 - The model `this-model-does-not-exist` does not exist or you do not have access to it.
[ask] attempt 2/3 on 'this-model-does-not-exist' failed -> OpenAIModelNotFoundError: Error code: 404 - The model `this-model-does-not-exist` does not exist or you do not have access to it.
SOP: New Employee Onboarding (Short)
[ask] attempt 1/3 on 'bad-1' failed -> OpenAIModelNotFoundError: Error code: 404 - The model `bad-1` does not exist or you do not have access to it.
[ask] attempt 2/3 on 'bad-1' failed -> OpenAIModelNotFoundError: Error code: 404 - The model `bad-1` does not exist or you do not have access to it.
[ask] attempt 3/3 on 'bad-2' failed -> OpenAIModelNotFoundError: Error code: 404 - The model `bad-2` does not exist or you do not have access to it.
Got LLMError as expected: All 3 attempts failed. Last error: OpenAIModelNotFoundError: Error code: 404 - The model `bad-2` does not exist 

## Recap

You made a model call, used messages, got a **Pydantic object** back with structured output, and wrapped every call in **retry and fallback**. The schemas (`Task`, `Plan`, `Section`) and `ask` are copied unchanged into `doc_agent.py`.

Next, Lab 2 forgets LLMs completely and learns the graph mechanics: state, reducers and the loop.

### Exercise


Write a fourth schema, `Review`, with `score: int` and `comments: list[str]`, and use `ask` to have the model review one section it wrote. This is the seed of a reflection step, which you could add later as a second improvement.